# Bag-of-Words GRPO

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.grpo import BagOfWordsGRPOConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/grpo.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsGRPOConfig.get_canonical(
    dataset="homoskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-grpo-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=4,
    gaussian_stdev=1.0,
)

display(config.visualize())

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

grpo epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

grpo epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,9600.457031,984.800781,49557.058594,49984.0,9600.457031,964.075928,1973.348999,49984.0,2556.008057,1382.705322,49494.257812,49920.0,2556.008057,1307.772583,1999.744507,49920.0
1,4661.839844,1854.777588,49548.230469,49984.0,4661.839844,1643.555176,1973.295166,49984.0,1946.558105,1383.78125,49494.257812,49920.0,1946.558105,1428.657104,1999.744507,49920.0


In [5]:
analysis = BagOfWordsAnalysisConfig.from_grouped({"example": [(0, config.study_folder)]})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])


In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
0.033691,0.015564,0.761719
0.021973,-0.263672,-0.055908
0.170898,0.0,-0.238281
-0.071777,0.046631,-0.679688
-0.22168,-0.279297,0.376953
